# Employee Data Analysis Project

**Tools:** Python, Pandas, NumPy, Matplotlib, Seaborn

## Objective
Explore a small employee-style dataset through data inspection, cleaning, descriptive statistics and visualisation. This is a descriptive EDA exercise, not a validated employee-performance or salary prediction model.

## Data limitations
The supplied data contains 50 records. Its original source, collection method, salary currency/unit, group definitions and scoring scales are not documented. Interpret findings as patterns in this supplied sample only. They do not establish causation or represent a wider workforce.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from io import StringIO

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)
print('Libraries imported successfully.')


## 1. Load the Employee Dataset

The notebook loads the committed `Employee Dataset.csv` when run from the repository. It retains an embedded fallback for standalone use and checks that both copies match.


In [ ]:
csv_data = 'id,groups,age,healthy_eating,active_lifestyle,salary\n0,A,36,5,5,2297\n1,A,26,3,5,1134\n2,A,61,8,1,4969\n3,O,24,3,6,902\n4,O,39,6,2,3574\n5,O,32,5,3,2761\n6,AB,25,4,6,1484\n7,B,39,8,6,3809\n8,A,32,5,6,2065\n9,A,28,4,8,1020\n10,A,30,6,9,1950\n11,O,40,7,1,4387\n12,O,23,3,2,1830\n13,O,30,6,8,2182\n14,AB,51,7,5,3460\n15,B,26,3,8,662\n16,A,30,3,5,1134\n17,A,22,4,8,1020\n18,A,27,2,6,779\n19,O,32,7,6,3228\n20,O,39,7,6,3228\n21,O,26,1,6,1360\n22,AB,40,7,5,3460\n23,B,37,6,4,3110\n24,A,28,5,8,1601\n25,A,30,6,7,2414\n26,A,62,9,6,700\n27,O,37,3,5,1134\n28,O,30,6,7,2414\n29,O,37,8,6,3809\n30,AB,28,4,7,1252\n31,B,30,7,8,2764\n32,A,22,3,8,662\n33,A,25,4,6,1484\n34,A,29,5,7,1833\n35,O,27,2,4,785\n36,O,39,5,5,2297\n37,O,26,3,5,1134\n38,AB,26,3,7,670\n39,B,29,2,6,779\n40,A,30,6,8,2182\n41,A,32,5,3,2761\n42,A,31,6,7,2414\n43,O,23,4,10,556\n44,O,35,6,6,2646\n45,O,25,5,8,1601\n46,AB,29,4,3,2179\n47,B,30,4,6,1484\n48,A,30,5,6,2065\n49,A,29,6,9,1950\n'

from pathlib import Path

embedded_df = pd.read_csv(StringIO(csv_data))
csv_path = Path("Employee Dataset.csv")
if csv_path.is_file():
    df = pd.read_csv(csv_path)
    pd.testing.assert_frame_equal(df, embedded_df, check_dtype=False)
    print("Loaded repository CSV and verified the embedded fallback matches.")
else:
    df = embedded_df.copy()
    print("Repository CSV not found. Loaded embedded fallback for standalone use.")

print("Dataset loaded successfully.")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
display(df.head())


## 2. Dataset Inspection

In [ ]:
print('Shape:', df.shape)
print('\nColumn names:')
print(df.columns.tolist())
print('\nData types:')
print(df.dtypes)


## 3. Data Cleaning

We standardise column names, validate the expected fields and data types, check missing values, inspect identifier uniqueness and remove exact duplicate rows. We do not invent missing values or assume undocumented score ranges.


In [ ]:
from src.employee_data import validate_employee_data# Validate required fields and basic domains without inventing undocumented score scales.validate_employee_data(df)print("Missing values:")display(df.isnull().sum().to_frame("Missing Values"))duplicate_count = int(df.duplicated().sum())print("Exact duplicate rows:", duplicate_count)if duplicate_count:    raise ValueError("Exact duplicate rows found. Review the source data before analysis.")print("Dataset validation passed.")print("Unique employee IDs:", df["id"].nunique())

## 4. Statistical Summary

In [ ]:
display(df.describe(include='all').T)


## 5. Numerical Data Analysis

The `id` field is an identifier, not a measured employee attribute. We exclude it from distribution plots and statistical interpretation.


In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns.drop('id', errors='ignore').tolist()
print('Numerical analysis columns:', numeric_cols)

for col in numeric_cols:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[col].dropna(), kde=True)
    plt.title(f'Distribution of {col}')
    plt.xlabel(col)
    plt.ylabel('Frequency')
    plt.tight_layout()
    plt.show()


## 6. Categorical Data Analysis

In [ ]:
categorical_cols = df.select_dtypes(exclude=np.number).columns.tolist()
print('Categorical columns:', categorical_cols)

for col in categorical_cols:
    plt.figure(figsize=(8,5))
    order = df[col].value_counts(dropna=False).index
    sns.countplot(data=df, x=col, order=order)
    plt.title(f'Distribution of {col}')
    plt.xlabel(col)
    plt.ylabel('Number of Employees')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()


## 7. Correlation Analysis

The identifier is excluded. Correlations describe associations within this small sample. They do not prove that age, lifestyle scores or any other variable causes salary differences. The meanings and scales of the lifestyle scores are not documented.


In [ ]:
if len(numeric_cols) >= 2:
    corr = df[numeric_cols].corr()
    display(corr)
    plt.figure(figsize=(9,6))
    sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0)
    plt.title('Correlation Heatmap')
    plt.tight_layout()
    plt.show()
else:
    print('At least two numerical columns are needed for correlation analysis.')


## 8. Automatic Key Findings

The summary reports descriptive values for measured fields only. It excludes the ID field from averages and ranges.


In [ ]:
print('KEY FINDINGS')
print('-' * 40)
print('Records:', len(df))
print('Variables:', len(df.columns))
print('Missing cells:', int(df.isnull().sum().sum()))
print('Exact duplicate rows remaining:', int(df.duplicated().sum()))

for col in numeric_cols:
    print(f'{col}: minimum={df[col].min()}, maximum={df[col].max()}, average={df[col].mean():.2f}')

for col in categorical_cols:
    print(f'{col}: {df[col].nunique(dropna=True)} unique categories')


## 9. Conclusion

The notebook performs descriptive EDA on 50 supplied records. It validates the expected schema, checks data quality, summarises measured fields, visualises numerical and categorical distributions, and examines correlations without treating them as causal effects.

The strongest observed sample correlation is between healthy_eating and salary (approximately 0.76). This is a descriptive association only. The dataset's source, salary unit, group definitions and score scales are undocumented, so no wider workforce or policy conclusion is justified.

This project is suitable as a beginner EDA exercise. It does not contain a defined prediction task, model training or evidence of employee-performance outcomes.
